# Local Moran's I: an interactive lab

This lab moves from one global number to a statistic for every unit. We will build a small ring-shaped neighbour graph, calculate Local Moran's I with conditional permutations, inspect the four scatterplot quadrants and apply a false-discovery-rate cutoff.

**You need:** Python, NumPy, Matplotlib, libpysal and esda. The ring is a synthetic graph, not a geographic map.

PySAL's default quadrant codes are 1=High–High, 2=Low–High, 3=Low–Low and 4=High–Low. The code below makes that convention explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import W
from libpysal.weights.spatial_lag import lag_spatial
from esda.moran import Moran_Local
from esda import fdr


## 1. Build a ring of connected units

Each unit is connected to the unit before and after it. The first and last units are also connected, making a ring. We assign higher values to one run of units and lower values to another.

In [ ]:
n = 12
neighbors = {i: [(i - 1) % n, (i + 1) % n] for i in range(n)}
w = W(neighbors)
w.transform = 'r'

values = np.array([10, 11, 9, 2, 1, 2, 8, 9, 10, 1, 2, 1], dtype=float)

local = Moran_Local(
    values,
    w,
    transformation='r',
    permutations=999,
    seed=12345,
    alternative='two-sided',
)

print('Number of units:', n)
print('Number of directed neighbour links:', sum(len(v) for v in neighbors.values()))
print('Local Moran values:', np.round(local.Is, 3))


In [ ]:
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
xy = np.column_stack([np.cos(angles), np.sin(angles)])

fig, ax = plt.subplots(figsize=(6, 6))
for i, linked in neighbors.items():
    for j in linked:
        if i < j:
            ax.plot([xy[i, 0], xy[j, 0]], [xy[i, 1], xy[j, 1]], linewidth=2)
ax.scatter(xy[:, 0], xy[:, 1], s=850, zorder=3)
for i, (x, y) in enumerate(xy):
    ax.text(x, y, f'U{i}\n{values[i]:g}', ha='center', va='center', color='white', weight='bold')
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Synthetic ring graph: unit ID and observed value')
plt.show()


## 2. Place each unit in a Moran scatterplot

The horizontal coordinate is the unit's standardised value. The vertical coordinate is the spatial lag of the standardised values. The four quadrants distinguish clusters from spatial outliers.

In [ ]:
z = local.z
spatial_lag = lag_spatial(w, z)
quadrants = {
    1: 'High–High',
    2: 'Low–High',
    3: 'Low–Low',
    4: 'High–Low',
}

fig, ax = plt.subplots(figsize=(7, 6))
for code, label in quadrants.items():
    mask = local.q == code
    ax.scatter(z[mask], spatial_lag[mask], s=90, label=label)
for i, (x, y) in enumerate(zip(z, spatial_lag)):
    ax.annotate(f'U{i}', (x, y), xytext=(5, 5), textcoords='offset points')
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel('Unit value (standardised)')
ax.set_ylabel('Neighbourhood lag (standardised)')
ax.set_title('Local Moran scatterplot')
ax.legend()
plt.show()


## 3. Distinguish a quadrant from a significant result

The quadrant code describes the direction of the local relationship. It does not establish that the relationship is statistically unusual. Local Moran uses conditional permutations: the focal unit's value is held fixed while the remaining values are shuffled.

In [ ]:
fdr_cutoff = fdr(local.p_sim, alpha=0.05)
passes_fdr = local.p_sim <= fdr_cutoff

print('FDR cutoff:', fdr_cutoff)
print('Units passing FDR:', np.flatnonzero(passes_fdr).tolist())
print('Unit | Local I | Quadrant | Conditional permutation p | Passes FDR')
for i in range(n):
    print(
        f'U{i:02d} | {local.Is[i]: .3f} | {quadrants[int(local.q[i])]:10s} | '
        f'{local.p_sim[i]:.3f} | {bool(passes_fdr[i])}'
    )


## 4. Change the pattern

Replace the values with an alternating high-low sequence. The graph stays the same. Re-run the calculation and scatterplot cells to see how the quadrant pattern changes.

In [ ]:
alternating = np.array([10, 1] * 6, dtype=float)
local_alternating = Moran_Local(
    alternating,
    w,
    transformation='r',
    permutations=999,
    seed=12345,
    alternative='two-sided',
)
print('Alternating values:', alternating.tolist())
print('Quadrant codes:', local_alternating.q.tolist())
print('Local Moran values:', np.round(local_alternating.Is, 3))


## What to take away

- Local Moran gives each spatial unit its own statistic.
- High–High and Low–Low describe clusters; High–Low and Low–High describe spatial outliers.
- A quadrant is not the same as a statistically significant result.
- Conditional permutations keep the focal unit's value fixed.
- Hundreds of local tests create a multiple-comparison problem; consider FDR and explain the correction.
- The graph and spatial weights define the relationships being tested.

Continue with the [Local Moran's I lesson](../algorithms/statistics/local-morans-i.md) and the [global autocorrelation learning path](../learning-paths/global-spatial-autocorrelation.md).

**Reference:** [PySAL's Local Moran guide](https://pysal.org/esda/stable/user-guide/local_spatial_autocorrelation.html) and [API reference](https://pysal.org/esda/stable/generated/esda.Moran_Local.html).